# 00 — EDA và tiền xử lý dữ liệu SecuraAI
Kiểm tra Windows-APT 2025 trước khi huấn luyện và giải thích lựa chọn feature.

In [ ]:
from pathlib import Path
import json, hashlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import platform, sklearn
import importlib
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
    average_precision_score, confusion_matrix, make_scorer)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.dummy import DummyClassifier
from securaai_ml.data import load_training_data
import securaai_ml.modeling as modeling_module
importlib.reload(modeling_module)  # tránh module cache cũ khi source vừa được cập nhật
from securaai_ml.modeling import (candidate_factories, decision_scores, parameter_grids,
    teaching_candidate_factories, teaching_parameter_grids)
from securaai_ml.schema import MODEL_FEATURES, TARGET_COLUMN, GROUP_COLUMN
from securaai_ml.train import temporal_train_validation_test

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data, audit = load_training_data(ROOT / 'data' / 'raw')
train, validation, test = temporal_train_validation_test(data)
X_train, y_train = train.loc[:, MODEL_FEATURES], train[TARGET_COLUMN]
X_validation, y_validation = validation.loc[:, MODEL_FEATURES], validation[TARGET_COLUMN]
groups = train[GROUP_COLUMN]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
audit

## 1. Kích thước và phân bố nhãn

In [ ]:
labels = data[TARGET_COLUMN].map({0:'BENIGN',1:'MALICIOUS_APT'})
display(labels.value_counts().to_frame('rows'))
labels.value_counts().plot.bar(rot=0, title='Label distribution'); plt.show()

## 2. Phân bố theo thời gian

In [ ]:
daily = data.groupby([GROUP_COLUMN,TARGET_COLUMN]).size().unstack(fill_value=0)
display(daily)
daily.plot(figsize=(13,5), title='Events by collection date'); plt.show()

## 3. Chia 60/20/20 theo thời gian
Giữ nguyên nhóm ngày để giảm leakage giữa các phiên thu thập liên quan.

In [ ]:
splits=pd.DataFrame({'rows':[len(train),len(validation),len(test)]},index=['Train','Validation','Test']); splits['ratio']=splits.rows/len(data); splits

## 4. Feature engineering
Dùng event/provider/channel/severity, basename tiến trình, protocol/status, giờ/ngày, after-hours, cờ IP private/present, port và cờ tồn tại command/hash. Không dùng dữ liệu tiết lộ trực tiếp nhãn.

In [ ]:
display(pd.Series(MODEL_FEATURES, name='model_feature').to_frame())
print('Missing values after canonical feature building:', int(data.loc[:, MODEL_FEATURES].isna().sum().sum()))

## 5. Giảm chiều
Không dùng PCA: dữ liệu phần lớn categorical sparse và feature đã chọn theo ngữ nghĩa bảo mật. OneHotEncoder dùng `min_frequency=2` để hạn chế category hiếm.